# Customer Churn Analysis & Customer Intelligence

A reproducible exploratory analysis project for customer churn, retention, subscription revenue, support complaints, and customer risk indicators.

> **Data note:** The original workbook/database is not included in this repository. If no source file is found, this notebook creates a clearly labelled **synthetic demo dataset** so the code can be tested end-to-end. Do not report demo-data findings as real business results.

## 1. Setup

Install dependencies from `requirements.txt` before running the notebook. The analysis is designed for a workbook named `customer_churn_data_raw.xlsx` (sheets: `customer`, `subscription`, `support`) or a SQLite database named `customer_churn.db` containing equivalent tables.

In [ ]:
from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 2. Load source data

Priority: SQLite database, then Excel workbook. If neither exists, use synthetic data for a reproducible smoke test. The expected tables are customer, subscription, and support.

In [ ]:
def make_demo_data(seed: int = RANDOM_SEED, n_customers: int = 250):
    """Create synthetic demo data; never represents real customers or business results."""
    rng = np.random.default_rng(seed)
    ids = np.arange(10001, 10001 + n_customers)
    states = rng.choice(["Delhi", "Maharashtra", "Karnataka", "Bihar", "Rajasthan"], n_customers)
    state_country = {"Delhi": "India", "Maharashtra": "India", "Karnataka": "India", "Bihar": "India", "Rajasthan": "India"}
    start = pd.Timestamp("2022-01-01") + pd.to_timedelta(rng.integers(0, 1300, n_customers), unit="D")
    churn = rng.random(n_customers) < 0.27
    cancellation = [pd.Timestamp("2026-10-01") - pd.to_timedelta(int(x), unit="D") if flag else pd.NaT
                    for x, flag in zip(rng.integers(0, 900, n_customers), churn)]
    plans = rng.choice(["Basic", "Standard", "Premium"], n_customers, p=[0.4, 0.38, 0.22])
    contracts = rng.choice(["Monthly", "Annual"], n_customers, p=[0.65, 0.35])
    customers = pd.DataFrame({
        "customerid": ids,
        "name": [f"Customer {i}" for i in ids],
        "dob": pd.to_datetime(rng.integers(1960, 2004, n_customers).astype(str) + "-01-01", errors="coerce"),
        "gender": rng.choice(["Male", "Female", "Men", "Women"], n_customers),
        "state": states,
        "country": [state_country[s] for s in states],
        "interests": rng.choice(["Sports", "Travel", "Technology", None], n_customers),
        "pincode": rng.integers(100000, 999999, n_customers),
    })
    subscriptions = pd.DataFrame({
        "customerid": ids,
        "subscription_start_date": start,
        "renewal_date": start + pd.to_timedelta(rng.integers(30, 365, n_customers), unit="D"),
        "cancellation_date": cancellation,
        "plan_type": plans,
        "contract_type": contracts,
        "monthly_charges": np.round(rng.uniform(400, 2500, n_customers), 2),
        "churn_score": rng.integers(0, 101, n_customers),
    })
    support_rows = []
    for cid in ids:
        for _ in range(int(rng.poisson(0.8))):
            support_rows.append({"customerid": cid,
                                 "complaint_date": pd.Timestamp("2026-10-01") - pd.to_timedelta(int(rng.integers(0, 900)), unit="D"),
                                 "escalations": rng.choice(["Y", "N", None], p=[0.18, 0.72, 0.10]),
                                 "col_1": None,
                                 "comment": "Synthetic demo complaint"})
    support = pd.DataFrame(support_rows, columns=["customerid", "complaint_date", "escalations", "col_1", "comment"])
    return {"customer": customers, "subscription": subscriptions, "support": support}


def load_source_tables():
    db_path = Path("customer_churn.db")
    excel_path = Path("customer_churn_data_raw.xlsx")
    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            available = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
            lookup = {name.lower(): name for name in available}
            required = ["customer", "subscription", "support"]
            missing = [name for name in required if name not in lookup]
            if missing:
                raise ValueError(f"SQLite database is missing required tables: {missing}. Found: {available}")
            return {name: pd.read_sql_query(f'SELECT * FROM "{lookup[name]}"', conn) for name in required}, "SQLite database"
    if excel_path.exists():
        xls = pd.ExcelFile(excel_path)
        lookup = {name.lower(): name for name in xls.sheet_names}
        required = ["customer", "subscription", "support"]
        missing = [name for name in required if name not in lookup]
        if missing:
            raise ValueError(f"Excel workbook is missing required sheets: {missing}. Found: {xls.sheet_names}")
        return {name: pd.read_excel(excel_path, sheet_name=lookup[name]) for name in required}, "Excel workbook"
    return make_demo_data(), "SYNTHETIC DEMO DATA (not real business data)"

source_tables, data_source = load_source_tables()
print(f"Data source: {data_source}")
for table_name, table in source_tables.items():
    print(f"{table_name}: {table.shape[0]:,} rows × {table.shape[1]:,} columns")
if data_source.startswith("SYNTHETIC"):
    print("WARNING: Results below are illustrative only. Add your real workbook/database to analyze real data.")

## 3. Validate and clean tables

Required columns are checked explicitly so that schema problems fail with a useful message instead of an unclear `KeyError`. Missing values are not silently treated as a negative escalation.

In [ ]:
df_customer = source_tables["customer"].copy()
df_subscription = source_tables["subscription"].copy()
df_support = source_tables["support"].copy()

required_columns = {
    "customer": {"customerid", "state"},
    "subscription": {"customerid", "subscription_start_date", "cancellation_date", "plan_type", "monthly_charges"},
    "support": {"customerid"},
}
for table_name, required in required_columns.items():
    actual = set(source_tables[table_name].columns)
    missing = sorted(required - actual)
    if missing:
        raise ValueError(f"Table '{table_name}' is missing required columns: {missing}. Available columns: {sorted(actual)}")

# Normalize column names for robust matching across source files.
for frame in (df_customer, df_subscription, df_support):
    frame.columns = [str(c).strip().lower() for c in frame.columns]

# Customer table cleaning.
if "name" in df_customer.columns and "customer_name" not in df_customer.columns:
    df_customer = df_customer.rename(columns={"name": "customer_name"})
for col in ["interests", "pincode"]:
    if col in df_customer.columns:
        df_customer = df_customer.drop(columns=col)
if "dob" in df_customer.columns:
    df_customer["dob"] = pd.to_datetime(df_customer["dob"], errors="coerce")
if "gender" in df_customer.columns:
    df_customer["gender"] = df_customer["gender"].replace({"Men": "Male", "Women": "Female"})
if {"state", "country"}.issubset(df_customer.columns):
    # Fill only when a state maps unambiguously to one known country.
    mapping_counts = df_customer.dropna(subset=["state", "country"]).groupby("state")["country"].nunique()
    unambiguous_states = mapping_counts[mapping_counts == 1].index
    state_country = (df_customer[df_customer["state"].isin(unambiguous_states)]
                     .dropna(subset=["state", "country"]).drop_duplicates("state").set_index("state")["country"])
    missing_country = df_customer["country"].isna()
    df_customer.loc[missing_country, "country"] = df_customer.loc[missing_country, "state"].map(state_country)

# Subscription dates and numeric fields.
for col in ["subscription_start_date", "renewal_date", "cancellation_date"]:
    if col in df_subscription.columns:
        df_subscription[col] = pd.to_datetime(df_subscription[col], errors="coerce")
for col in ["monthly_charges", "churn_score"]:
    if col in df_subscription.columns:
        df_subscription[col] = pd.to_numeric(df_subscription[col], errors="coerce")

# Support dates, optional columns, and per-customer aggregation.
if "complaint_date" in df_support.columns:
    df_support["complaint_date"] = pd.to_datetime(df_support["complaint_date"], errors="coerce")
for col in ["col_1", "comment"]:
    if col in df_support.columns:
        df_support = df_support.drop(columns=col)
if "escalations" in df_support.columns:
    df_support["escalations"] = df_support["escalations"].astype("string").str.strip().str.upper()
    df_support.loc[~df_support["escalations"].isin(["Y", "N"]), "escalations"] = pd.NA

print("Duplicate customer IDs:", int(df_customer["customerid"].duplicated().sum()))
print("Duplicate subscription IDs:", int(df_subscription["customerid"].duplicated().sum()))
print("Support rows:", len(df_support))
display(df_customer.head(), df_subscription.head(), df_support.head())

## 4. Aggregate support before joining

A direct join to raw support records can multiply subscription rows and inflate revenue/churn metrics. Support is aggregated to one row per customer before joining.

In [ ]:
if df_customer["customerid"].duplicated().any():
    raise ValueError("Customer table has duplicate customerid values. Resolve duplicates before analysis.")
if df_subscription["customerid"].duplicated().any():
    raise ValueError("Subscription table has multiple rows per customerid. This notebook expects one active/latest subscription record per customer.")

if df_support.empty:
    support_summary = pd.DataFrame(columns=["customerid", "complaint_count", "escalation_status"])
else:
    support_summary = df_support.groupby("customerid", dropna=False).agg(
        complaint_count=("customerid", "size"),
        escalation_status=("escalations", lambda s: "Y" if (s == "Y").any() else ("N" if (s == "N").any() else pd.NA))
    ).reset_index()

# Keep all subscription customers; enrich with customer/support information.
df = (df_subscription
      .merge(df_customer, on="customerid", how="left", validate="one_to_one", suffixes=("", "_customer"))
      .merge(support_summary, on="customerid", how="left", validate="one_to_one"))
df["complaint_count"] = pd.to_numeric(df["complaint_count"], errors="coerce").fillna(0).astype(int)
df["churn_flag"] = df["cancellation_date"].notna().astype(int)

# Validate the join did not duplicate subscription rows.
assert len(df) == len(df_subscription), "Unexpected row multiplication after merge. Check source-table uniqueness."
print(f"Merged customer-level dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

## 5. Data quality overview

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(quality)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Unique customers:", int(df["customerid"].nunique()))

## 6. Key business metrics

- **Churn rate:** customers with a cancellation date divided by all subscription customers.
- **Retention rate:** 100% minus churn rate for this customer snapshot.
- **Revenue at risk:** sum of monthly charges for churned customers. This is a monthly-revenue proxy, not cumulative lifetime revenue.
- **Escalation rate:** calculated only among customers with a known Y/N status.

In [ ]:
customer_count = df["customerid"].nunique()
churn_rate = df["churn_flag"].mean() * 100 if len(df) else np.nan
retention_rate = 100 - churn_rate if pd.notna(churn_rate) else np.nan
arpu = df["monthly_charges"].mean() if "monthly_charges" in df else np.nan
revenue_at_risk = df.loc[df["churn_flag"].eq(1), "monthly_charges"].sum(min_count=1)

as_of_date = pd.Timestamp.today().normalize()
end_date = df["cancellation_date"].where(df["cancellation_date"].notna(), as_of_date)
df["tenure_days"] = (end_date - df["subscription_start_date"]).dt.days
df.loc[df["tenure_days"] < 0, "tenure_days"] = np.nan
avg_tenure_days = df["tenure_days"].mean()

known_escalation = df["escalation_status"].isin(["Y", "N"])
escalation_rate = (df.loc[known_escalation, "escalation_status"].eq("Y").mean() * 100
                   if known_escalation.any() else np.nan)
avg_complaints = df["complaint_count"].mean() if customer_count else np.nan

kpis = pd.DataFrame({"Metric": ["Customers", "Churn rate (%)", "Retention rate (%)", "ARPU (monthly charges)",
                                 "Revenue at risk (monthly charges)", "Average tenure (days)",
                                 "Escalation rate among known statuses (%)", "Average complaint records per customer"],
                     "Value": [customer_count, round(churn_rate, 2), round(retention_rate, 2),
                               round(arpu, 2) if pd.notna(arpu) else np.nan,
                               round(revenue_at_risk, 2) if pd.notna(revenue_at_risk) else np.nan,
                               round(avg_tenure_days, 1) if pd.notna(avg_tenure_days) else np.nan,
                               round(escalation_rate, 2) if pd.notna(escalation_rate) else np.nan,
                               round(avg_complaints, 2) if pd.notna(avg_complaints) else np.nan]})
display(kpis)

## 7. Churn by customer segments

In [ ]:
def churn_summary(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if group_col not in data.columns:
        print(f"Skipping '{group_col}': column not present in source data.")
        return pd.DataFrame()
    result = (data.groupby(group_col, dropna=False).agg(
        customers=("customerid", "nunique"),
        churn_rate_pct=("churn_flag", "mean"),
        monthly_charges=("monthly_charges", "sum")
    ).reset_index())
    result["churn_rate_pct"] = (result["churn_rate_pct"] * 100).round(2)
    return result.sort_values("churn_rate_pct", ascending=False)

churn_by_plan = churn_summary(df, "plan_type")
display(churn_by_plan)
if "state" in df.columns:
    display(churn_summary(df, "state"))
if "contract_type" in df.columns:
    display(churn_summary(df, "contract_type"))

## 8. Visualizations

In [ ]:
# Monthly churn trend: use cancellation date only for churned customers.
if "cancellation_date" in df.columns:
    trend_data = df.loc[df["churn_flag"].eq(1) & df["cancellation_date"].notna()].copy()
    if not trend_data.empty:
        trend_data["cancellation_month"] = trend_data["cancellation_date"].dt.to_period("M").astype(str)
        monthly_churn = trend_data.groupby("cancellation_month").size().sort_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(monthly_churn.index, monthly_churn.values, marker="o", linewidth=2)
        ax.set(title="Monthly Churn Trend", xlabel="Cancellation month", ylabel="Churned customers")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout(); plt.show()
    else:
        print("Monthly churn trend unavailable: no valid cancellation dates.")

if "plan_type" in df.columns and not df.empty:
    by_plan = df.groupby("plan_type", dropna=False)["churn_flag"].mean().mul(100).sort_values(ascending=False)
    ax = by_plan.plot(kind="bar", figsize=(7, 4), ylim=(0, 100), title="Churn Rate by Plan Type")
    ax.set_xlabel("Plan type"); ax.set_ylabel("Churn rate (%)"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

if "state" in df.columns and not df.empty:
    by_state = df.groupby("state", dropna=False)["churn_flag"].mean().mul(100).sort_values(ascending=False)
    ax = by_state.plot(kind="bar", figsize=(10, 4), title="Churn Rate by State")
    ax.set_xlabel("State"); ax.set_ylabel("Churn rate (%)"); plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()

## 9. Escalation and churn association

This is a simple association, not evidence that escalation causes churn. Unknown escalation statuses remain missing.

In [ ]:
df["escalation_flag"] = df["escalation_status"].map({"Y": 1, "N": 0})
correlation_data = df[["escalation_flag", "churn_flag"]].dropna()
if len(correlation_data) >= 2 and correlation_data["escalation_flag"].nunique() > 1 and correlation_data["churn_flag"].nunique() > 1:
    correlation = correlation_data["escalation_flag"].corr(correlation_data["churn_flag"])
    print(f"Pearson correlation (known escalation statuses only): {correlation:.3f}")
else:
    print("Correlation unavailable: insufficient valid records or variation.")
print("Records included:", len(correlation_data))

if "churn_score" in df.columns:
    df["churn_risk"] = pd.cut(df["churn_score"], bins=[-np.inf, 50, 70, np.inf],
                              labels=["Low", "Medium", "High"], right=False)
    display(df.groupby("churn_risk", observed=False, dropna=False).agg(
        customers=("customerid", "nunique"), churn_rate_pct=("churn_flag", lambda x: round(x.mean() * 100, 2)),
        avg_monthly_charges=("monthly_charges", "mean")).reset_index())

## 10. Exploratory correlation heatmap

Nominal categories such as plan and contract are not converted into arbitrary integer codes here. The heatmap uses numeric measures only, reducing the risk of misleading correlations.

In [ ]:
numeric_candidates = [c for c in ["monthly_charges", "churn_score", "tenure_days", "complaint_count", "churn_flag", "escalation_flag"] if c in df.columns]
correlation_matrix = df[numeric_candidates].corr(numeric_only=True)
if not correlation_matrix.empty:
    plt.figure(figsize=(8, 5))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title("Exploratory Correlation Heatmap (Numeric Features)")
    plt.tight_layout(); plt.show()
else:
    print("No numeric columns available for a correlation heatmap.")

## 11. Summary tables and export

In [ ]:
pivot_by_plan = pd.pivot_table(
    df, index="plan_type", values=["monthly_charges", "customerid", "churn_flag"],
    aggfunc={"monthly_charges": "sum", "customerid": "nunique", "churn_flag": "mean"},
    observed=False
).rename(columns={"customerid": "customer_count", "churn_flag": "churn_rate"})
pivot_by_plan["churn_rate"] = pivot_by_plan["churn_rate"] * 100
display(pivot_by_plan)

output_path = Path("exported_churn_data.csv")
df.to_csv(output_path, index=False)
print(f"Exported merged analysis data to: {output_path.resolve()}")

## 12. Interpretation checklist

Before publishing business conclusions, verify:

1. The notebook printed the expected real data source rather than synthetic demo data.
2. Customer IDs are unique in customer and subscription tables, and the merge row-count assertion passed.
3. Currency/units for `monthly_charges` are documented in the source data.
4. `cancellation_date` accurately represents churn and the snapshot date is appropriate.
5. Segment-level churn rates are interpreted with customer counts, not percentages alone.
6. Correlation is described as association, not causation.

## Limitations

This is an exploratory churn analysis, not a predictive machine-learning model. A production churn model needs a defined prediction date, leakage-safe features, train/test validation, and evaluation metrics. The synthetic fallback exists only to test code execution.